In [1]:
from inference_funcs import load_bit_phoneme_model, evaluate_model, decode_outputs
from dataset import getDatasetLoaders
import numpy as np
from edit_distance import SequenceMatcher
from typing import Any, Iterable, Sequence
import re 
from g2p_en import G2p
import numpy as np
g2p = G2p()

/home/ubuntu/miniconda/envs/speech-bci/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def postprocess_topk(topk_labels_indices_to_keep: np.ndarray, blank_token: str = "~") -> np.ndarray:
    """
    For each row:
    - If the blank token is present, remove it.
    - Otherwise, remove the last element.
    
    Always returns an array with one fewer column than input.
    """
    processed_rows = []
    for row in topk_labels_indices_to_keep:
        if blank_token in row:
            # remove first occurrence of blank
            new_row = [tok for tok in row if tok != blank_token]
        else:
            new_row = row[:-1]
        processed_rows.append(new_row)
    return np.array(processed_rows, dtype=object)



def topk_labels(logits: np.ndarray, K: int, vocab: list, apply_ctc_rule: bool) -> np.ndarray:
    """
    Args:
        logits: np.ndarray of shape (T, N) where
                N = 1 + len(vocab) (index 0 = CTC blank, rest follow vocab order)
        K: number of top tokens to return
        vocab: list of labels (phonemes or characters)
        apply_ctc_rule: removes blanks and repeats not separated by a blank

    Returns:
        np.ndarray of shape (T, K) with label strings
    """
    # Build id -> label mapping
    
    blank_token = "~"
    
    id2label = [blank_token] + vocab

    # Get top-K indices per timestep
    topk_ids = np.argsort(logits, axis=1)[:, -(K+1):][:, ::-1]

    # Map to labels
    topk_labels = np.vectorize(lambda i: id2label[i])(topk_ids)
    
    top1_ids = np.argsort(logits, axis=1)[:, -1:][:, ::-1]
    
    blank_indices = np.argwhere(top1_ids == 0).squeeze()
    
    # returns indices where the next index is the same
    repeating_indices = np.argwhere(top1_ids[:-1] == top1_ids[1:]).squeeze()
    
    indices_to_remove = np.union1d(blank_indices, repeating_indices+1).squeeze()
    
    indices_to_keep = np.setdiff1d(np.arange(len(top1_ids)), indices_to_remove)
    
    topk_labels_indices_to_keep = postprocess_topk(topk_labels[indices_to_keep])
    
    # remove the blank token for rows which have it, for rows which don't have it remove the last element for topk_labels_indices_to_keep

    
    return topk_labels_indices_to_keep, indices_to_remove.shape[0], indices_to_keep.shape[0], indices_to_keep




def rows_to_text_dual(
    arr1: Any,
    arr2: Any,
    elem_sep: str = " ",
    row_sep: str = "\n",
    side_sep: str = " | "
) -> str:
    """
    Convert two 2-D arrays (or list-of-lists) to a single string with side-by-side rows.
    - Each element in a row is joined with `elem_sep`.
    - Each pair of rows is joined with `side_sep`.
    - Each line is joined with `row_sep` (newline by default).
    """
    # Support numpy arrays or list-of-lists
    rows1 = arr1.tolist() if isinstance(arr1, np.ndarray) else arr1
    rows2 = arr2.tolist() if isinstance(arr2, np.ndarray) else arr2
    
    if len(rows1) != len(rows2):
        raise ValueError("Both arrays must have the same number of rows")
    
    lines = []
    for r1, r2 in zip(rows1, rows2):
        if isinstance(r1, np.ndarray): r1 = r1.tolist()
        if isinstance(r2, np.ndarray): r2 = r2.tolist()
        left = elem_sep.join(map(str, r1))
        right = elem_sep.join(map(str, r2))
        lines.append(left + side_sep + right)
    
    return row_sep.join(lines)


def rows_to_text(arr: Any, elem_sep: str = " ", row_sep: str = "\n") -> str:
    """
    Convert a 2-D numpy array (or list of lists) to a single string.
    - Each element in a row is joined with `elem_sep`.
    - Each row is joined with `row_sep` (newline by default).
    """
    # Support ndarray or list-of-lists
    rows: Iterable = arr.tolist() if isinstance(arr, np.ndarray) else arr
    
    lines = []
    for r in rows:
        if isinstance(r, np.ndarray):
            r = r.tolist()
        lines.append(elem_sep.join(map(str, r)))
    
    return row_sep.join(lines)

def grapheme_to_phoneme(sentence):
    
    phonemes = ""
    
    for p in g2p(sentence):
        
        if p==' ':
            phonemes += p
        p = re.sub(r'[0-9]', '', p)  # Remove stress
        
        if re.match(r'[A-Z]+', p):  # Only keep phonemes
            phonemes += p
    
    return phonemes


In [3]:
# Phone definitions and mappings
PHONE_DEF = [
    'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B',  'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH'
]
PHONE_DEF_SIL = PHONE_DEF + ["<>"]


In [10]:
datasets = [f'ptDecoder_ctc_both_fold_number_{i}' for i in range(5)]
models = [f'time_masked_transfomer_fold_{i}_seed_0' for i in range(5)]

dataset_base_path = '/data2/neural_data/'
model_base_path = '/data2/models/cross_validation_with_transformer/'

partition = 'test'

device = 'cuda'

phoneme_outputs = []
phoneme_frames_removed = 0
phoneme_frames_kept = 0
phoneme_indices_kept = []

ground_truth_sentences_arr = []
decoded_sentences_arr = []

for fn, (dataset, model) in enumerate(zip(datasets, models)):

    data_file = f"{dataset_base_path}{dataset}"

    trainLoaders, testLoaders, loadedData = getDatasetLoaders(
            data_file, 8, None, 
            False
    )
    
    bit_phoneme_filepath = f"{model_base_path}{model}"
    model, args = load_bit_phoneme_model(bit_phoneme_filepath)
    model = model.to(device)
    
    outputs, cer, per_day_cer = evaluate_model(model, loadedData, args, partition=partition, device='cuda')
    
    decoded_strs = decode_outputs(outputs['decodedSeqs'], mode='phoneme')
    true_seq_strs = outputs['transcriptions']
    
    ground_truth_sentences_arr.extend(true_seq_strs)
    decoded_sentences_arr.extend(decoded_strs)
    
    print(f"Fold number {fn} with {len(decoded_strs)} val sentences.")
    
    for phoneme_output in outputs['logits']:
        
        topk_phones, pfr, pfk, _ = topk_labels(phoneme_output, K=10, vocab=PHONE_DEF_SIL, apply_ctc_rule=True)
        phoneme_outputs.append(topk_phones)
        phoneme_frames_removed += pfr
        phoneme_frames_kept += pfk    
                

CER DAY 0: 0.379965
CER DAY 1: 0.313458
CER DAY 2: 0.234653
CER DAY 3: 0.250000
CER DAY 4: 0.142358
CER DAY 5: 0.132031
CER DAY 6: 0.124710
CER DAY 7: 0.128391
CER DAY 8: 0.124509
CER DAY 9: 0.150767
CER DAY 10: 0.142025
CER DAY 11: 0.159322
CER DAY 12: 0.128820
CER DAY 13: 0.127837
CER DAY 14: 0.127565
CER DAY 15: 0.134791
CER DAY 16: 0.124525
CER DAY 17: 0.119485
CER DAY 18: 0.137654
CER DAY 19: 0.144868
CER DAY 20: 0.132697
CER DAY 21: 0.133440
CER DAY 22: 0.154793
CER DAY 23: 0.154559
Model performance (CER): 0.16854516179296733
Fold number 0 with 2000 val sentences.
CER DAY 0: 0.320672
CER DAY 1: 0.287336
CER DAY 2: 0.213750
CER DAY 3: 0.253488
CER DAY 4: 0.103928
CER DAY 5: 0.102278
CER DAY 6: 0.145192
CER DAY 7: 0.148489
CER DAY 8: 0.140452
CER DAY 9: 0.147083
CER DAY 10: 0.142300
CER DAY 11: 0.154139
CER DAY 12: 0.145225
CER DAY 13: 0.143548
CER DAY 14: 0.118660
CER DAY 15: 0.122088
CER DAY 16: 0.109971
CER DAY 17: 0.126066
CER DAY 18: 0.134126
CER DAY 19: 0.146353
CER DAY 20: 

In [20]:
datasets = ['ptDecoder_ctc_both']
models = ["transformer_short_training_fixed_seed_0"]

dataset_base_path = '/data2/neural_data/'
model_base_path = '/data2/models/'

partition = 'train'

device = 'cuda'

phoneme_outputs_train = []
phoneme_frames_removed_train = 0
phoneme_frames_kept_train = 0
phoneme_indices_kept_train = []

ground_truth_sentences_arr_train = []
decoded_sentences_arr_train = []

for fn, (dataset, model) in enumerate(zip(datasets, models)):

    data_file = f"{dataset_base_path}{dataset}"

    trainLoaders, testLoaders, loadedData = getDatasetLoaders(
            data_file, 8, None, 
            False
    )
    
    bit_phoneme_filepath = f"{model_base_path}{model}"
    model, args = load_bit_phoneme_model(bit_phoneme_filepath)
    
    
    model = model.to(device)
    
    outputs, cer, per_day_cer = evaluate_model(model, loadedData, args, partition=partition, device='cuda')
    
    decoded_strs_train = decode_outputs(outputs['decodedSeqs'], mode='phoneme')
    true_seq_strs_train = outputs['transcriptions']
        
    for phoneme_output in outputs['logits']:
        
        topk_phones, pfr, pfk, _ = topk_labels(phoneme_output, K=10, vocab=PHONE_DEF_SIL, apply_ctc_rule=True)
        phoneme_outputs_train.append(topk_phones)
        phoneme_frames_removed_train += pfr
        phoneme_frames_kept_train += pfk    

CER DAY 0: 0.128099
CER DAY 1: 0.110004
CER DAY 2: 0.037424
CER DAY 3: 0.051232
CER DAY 4: 0.014383
CER DAY 5: 0.016164
CER DAY 6: 0.015789
CER DAY 7: 0.014865
CER DAY 8: 0.021737
CER DAY 9: 0.022946
CER DAY 10: 0.017699
CER DAY 11: 0.019985
CER DAY 12: 0.025427
CER DAY 13: 0.018895
CER DAY 14: 0.011715
CER DAY 15: 0.013761
CER DAY 16: 0.012803
CER DAY 17: 0.017164
CER DAY 18: 0.015778
CER DAY 19: 0.013170
CER DAY 20: 0.013950
CER DAY 21: 0.012922
CER DAY 22: 0.016573
CER DAY 23: 0.020801
Model performance (CER): 0.030371403732233477
Fold number 0 with 840 sentences.


In [55]:
true_seq_strs_train_sorted == ground_truth_sentences_arr_sorted

True

In [70]:
sorted_idxs_train = np.argsort(true_seq_strs_train)
sorted_idxs_cross_val = np.argsort(ground_truth_sentences_arr)

true_seq_strs_train_sorted = [true_seq_strs_train[i] for i in sorted_idxs_train]
ground_truth_sentences_arr_sorted = [ground_truth_sentences_arr[i] for i in sorted_idxs_cross_val]

phoneme_outputs_sorted = [phoneme_outputs[i] for i in sorted_idxs_cross_val]
phoneme_outputs_train_sorted = [phoneme_outputs_train[i] for i in sorted_idxs_train]

import random 
random.seed(42)
shuffled_indices = list(range(len(phoneme_outputs_sorted)))
random.shuffle(shuffled_indices)

# these two lists will be the same
true_seq_strs_train_shuffled = [true_seq_strs_train_sorted[i] for i in shuffled_indices]
ground_truth_sentences_arr_shuffled = [ground_truth_sentences_arr_sorted[i] for i in shuffled_indices]

phoneme_outputs_shuffled = [phoneme_outputs_sorted[i] for i in shuffled_indices]
phoneme_outputs_train_shuffled = [phoneme_outputs_train_sorted[i] for i in shuffled_indices]

# alternate between high accuracy and lower accuracy sentences
block_size = 800
combined_ground_truth = []
combined_phoneme_outputs = []
for i in range(0, 8800, block_size):
    
    combined_ground_truth.extend(true_seq_strs_train_shuffled[i:i+block_size])
    combined_ground_truth.extend(ground_truth_sentences_arr_shuffled[i:i+block_size])
    
    combined_phoneme_outputs.extend(phoneme_outputs_train_shuffled[i:i+block_size])
    combined_phoneme_outputs.extend(phoneme_outputs_shuffled[i:i+block_size])
    
print(len(combined_phoneme_outputs))
print(len(combined_ground_truth))


17600
17600


In [71]:
compute_per_sentence_cer = False
cer_per_sentence = []
if compute_per_sentence_cer:
    for true_seq, decoded in zip(outputs['trueSeqs2'], outputs['decodedSeqs2']):
        
        ed = SequenceMatcher(a=true_seq.tolist(), b=decoded.tolist()).distance()
        cer_per_sentence.append(ed/len(true_seq))
        
    np.save('/data2/perf_metrics/cer_per_sentence', cer_per_sentence)

In [79]:
import json
from pathlib import Path

OUT_JSONL = '/datata2/jsonl/cross_val_and_train.jsonl'
eval_mode = True

USER_HDR = "<|start_header_id|>user<|end_header_id|>"
ASST_HDR = "<|start_header_id|>assistant<|end_header_id|>"
EOT      = "<|eot_id|>"   # include if your format expects it

prompt = (
    'You are helping decode speech from neural activity to help restore communication for a paralyzed patient. '
    'For each time bin of neural activity, a neural network model provides the 10 most probable tokens. '
    'Tokens consist of ARPAbet phonemes and the space character, denoted as <>. '
    'On each line, the the top 10 tokens are listed in order, from most to least likely. '
    'Each separate line represents the model output for a given non-overlapping neural time bin, starting from the beginning of the text. '
    'Since the model output is not perfectly accurate, your job is to correct its output by producing the ground-truth phoneme sequence along with the corresponding ground-truth word-level sentence. '
    'Produce coherent text that is gramatically correct. '
    'Output only the corrected phoneme sequence and word-level sentence, no additional explanations or metadata.'
)

with Path(OUT_JSONL).open("w", encoding="utf-8") as fout:
    
    for idx in range(len(combined_phoneme_outputs)):
        
        topk_phones = rows_to_text(combined_phoneme_outputs[idx])
        
        if eval_mode:
            ground_truth = ""
            
        else:
            gt_sentence = combined_ground_truth[idx]
            ground_truth = f"{gt_sentence}{EOT}"
        
        msg = (
            f"{USER_HDR}\n\n"
            f"{prompt}\n\n"
            f"{topk_phones}\n\n"
            f"{ASST_HDR}\n\n"
            f"{ground_truth}"
        )
        
        obj = {"text": msg}
        
        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [78]:
print(msg)

<|start_header_id|>user<|end_header_id|>

You are helping decode speech from neural activity to help restore communication for a paralyzed patient. For each time bin of neural activity, a neural network model provides the 10 most probable tokens. Tokens consist of ARPAbet phonemes and the space character, denoted as <>. On each line, the the top 10 tokens are listed in order, from most to least likely. Each separate line represents the model output for a given non-overlapping neural time bin, starting from the beginning of the text. Since the model output is not perfectly accurate, your job is to correct its output by producing the ground-truth phoneme sequence along with the corresponding ground-truth word-level sentence. Produce coherent text that is gramatically correct. Output only the corrected phoneme sequence and word-level sentence, no additional explanations or metadata.

DH S D <> TH N L T AH Y
AH IH DH OW UW EH IY S ER EY
<> AH Z D N S R IH T ER
V IH <> R F D AH ER IY N
IH Z